### the intention here is to build the workflow where we gather data response from the llm about whats in the pdf guia, that was proven before that it can be accurate, in notebook 04, and then compare with the field DS_PROCEDIMENTO and see if they match or not, generating another df with a aut_validacao_status. we can t keep ds contato , so in the workflow we check if theres a analysis form the analist or not for comparing only purposes of the past data

## Part 0: getting the right avisos from enriched excel file.

In [7]:
# download file from s3 bucket = ,S3_BUCKET_NAME="agente-ai-laudos" , folder = segunda_saida/, file_name = "consolidado_enriquecido.xlsx"
# excel_handler = S3ExcelClient(homolog=False)


downlaod excel file

In [ ]:
# ...

extract avisos from eexcel

In [70]:
import pandas as pd
from typing import List, Tuple, Optional

def extract_avisos_from_tab(
    xls: pd.ExcelFile, 
    tab_name: str, 
    mask_criteria: dict, 
    limit: Optional[int] = None
) -> Tuple[List[int], str]:
    """
    Extract avisos from a specific Excel tab based on mask criteria and limit.
    
    Args:
        xls (pd.ExcelFile): Excel file object
        tab_name (str): Name of the Excel sheet/tab
        mask_criteria (dict): Dictionary with column name and values for filtering
        limit (int, optional): Maximum number of avisos to extract (takes last N records)
        
    Returns:
        Tuple[List[int], str]: Tuple containing:
            - List of aviso IDs (CD_AVISO_CIRURGIA)
            - String representation of avisos for SQL queries
    """
    # Read the specified tab
    df = pd.read_excel(xls, sheet_name=tab_name)
    
    # Apply mask criteria
    mask = pd.Series([True] * len(df))  # Start with all True
    
    for column, values in mask_criteria.items():
        if column in df.columns:
            if isinstance(values, set):
                mask = mask & df[column].isin(values)
            else:
                mask = mask & (df[column] == values)
    
    # Filter the dataframe
    filtered_avisos = df.loc[mask, "CD_AVISO_CIRURGIA"]
    
    # Apply limit if specified (take last N records)
    if limit and len(filtered_avisos) > limit:
        filtered_avisos = filtered_avisos.tail(limit)
    
    # Convert to list
    avisos_list = filtered_avisos.tolist()
    
    # Create string representation for SQL queries
    avisos_str = ", ".join(str(i) for i in avisos_list)
    
    print(f"📋 Found {len(avisos_list)} records to process for {tab_name}")
    if len(avisos_list) == 0:
        print(f"   ℹ️  No matching records found in {tab_name}")
    
    return avisos_list, avisos_str

In [74]:


# Dictionary to hold the results for each file
filtered_avisos = {}
# temporary code because we cant acess s3 right now
excel_path = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/excel/consolidado_enriquecido.xlsx"

xls = pd.ExcelFile(excel_path)

# Read both tabs for reference
df_aut = pd.read_excel(xls, sheet_name="Autorizados")
df_pendente = pd.read_excel(xls, sheet_name="Pendentes")

# Define criteria for filtering
valid_friendly = {"Contorno", "Betim-Contagem", "Salvador/Bahia"}
mask_criteria = {"friendly_name": valid_friendly}

# Extract avisos from both tabs using the reusable function
avisos_interest_list_aut, avisos_interest_list_aut_str = extract_avisos_from_tab(
    xls=xls,
    tab_name="Autorizados", 
    mask_criteria=mask_criteria,
    limit=15  # No limit for now, but you can set a number like limit=100
)

avisos_interest_list_pendente, avisos_interest_list_pendente_str = extract_avisos_from_tab(
    xls=xls,
    tab_name="Pendentes", 
    mask_criteria=mask_criteria,
    limit=15  # No limit for now, but you can set a number like limit=50
)

# Combine both lists
avisos_interest_list_combined = avisos_interest_list_aut + avisos_interest_list_pendente
avisos_interest_list_str = ", ".join(str(i) for i in avisos_interest_list_combined)

print(f"📋 Found {len(avisos_interest_list_aut)} records in Autorizados")
print(f"📋 Found {len(avisos_interest_list_pendente)} records in Pendentes") 
print(f"📋 Total avisos to process: {len(avisos_interest_list_combined)}")
print(f"📋 String length: {len(avisos_interest_list_str)} characters")
print(f"✅ Verification: {len(avisos_interest_list_combined)} == {len(avisos_interest_list_aut)} + {len(avisos_interest_list_pendente)} = {len(avisos_interest_list_aut) + len(avisos_interest_list_pendente)}")

📋 Found 15 records to process for Autorizados
📋 Found 15 records to process for Pendentes
📋 Found 15 records in Autorizados
📋 Found 15 records in Pendentes
📋 Total avisos to process: 30
📋 String length: 238 characters
✅ Verification: 30 == 15 + 15 = 30


In [65]:
# check if some avisos in a list also is in another
for aviso in avisos_interest_list_aut:
    if aviso in avisos_interest_list_pendente:
        print(f"   📋 Aviso {aviso} found in both lists")
for aviso in avisos_interest_list_pendente:
    if aviso in avisos_interest_list_aut:
        print(f"   📋 Aviso {aviso} found in both lists")

theres no overlapping avisos

# Part 1

load the dataframe into memory and bring it to a valid input for unstructured

### DS_CONTATO, meaning the data from query finalizado, we dont care in production stage anymore, so we dont use this base anymore

In [66]:
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.services.database.oracle import OracleService
from app.services.database.mariadb import MariaDBService
from app.utils.db_operations import load_query_from_file, execute_query_to_df
from app.utils.config import load_config
from app.utils.logger import get_logger


logger = get_logger(name=__name__)
config_vars = load_config()

mount the query string

In [10]:
# load the query strings
autorizacao_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_autorizacao.sql"
autorizacao_query_str = load_query_from_file(autorizacao_query_file)
autorizacao_sql_final = autorizacao_query_str.replace("?", avisos_interest_list_str, 1)

procedimento_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_procedimento.sql"
procedimento_query_str = load_query_from_file(procedimento_query_file)
procedimento_sql_final = procedimento_query_str.replace("?", avisos_interest_list_str, 1)

finalizado_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_pedido_finalizado_resposta.sql"
finalizado_query_str = load_query_from_file(finalizado_query_file)
finalizado_sql_final = finalizado_query_str.replace("?", avisos_interest_list_str, 1)

aviso_cirurgia_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_aviso_cirurgia.sql"
aviso_cirurgia_query_str = load_query_from_file(aviso_cirurgia_query_file)
aviso_cirurgia_sql_final = aviso_cirurgia_query_str.replace("?", avisos_interest_list_str, 1)

In [11]:
autorizacao_sql_final

'SELECT guia.cd_guia,\n       guia.tp_guia,\n       guia.cd_aviso_cirurgia,\n       guia.tp_situacao,\n       guia.dt_solicitacao,\n       guia.dt_autorizacao,\n       overmind_aux_log_pre.cd_guia,\n       overmind_aux_log_pre.cd_aviso,\n       overmind_aux_log_pre.dt_insercao,\n       overmind_aux_log_pre.dt_overmind,\n       overmind_aux_log_pre.ds_protocolo,\n       overmind_aux_log_pre.dt_status,\n       overmind_aux_log_pre.tp_status,\n       overmind_aux_log_pre.ds_status,\n       overmind_aux_log_pre.cd_senha,\n       overmind_aux_log_pre.dt_senha,\n       overmind_aux_log_pre.ds_guia_path   \n       FROM dbamv.guia\n       LEFT JOIN dbahmd.overmind_aux_log_pre ON overmind_aux_log_pre.cd_guia = guia.cd_guia\n          WHERE guia.cd_aviso_cirurgia IN (862520, 864591, 864794, 864853, 865102, 866143, 866435, 867288, 868647, 869083, 869319, 870620, 872333, 872778, 872924, 873656, 873801)'

In [12]:
maria_db_procedimento_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=procedimento_sql_final)
oracle_db_autorizacao_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=autorizacao_sql_final)
oracle_db_finalizado_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=finalizado_sql_final)
maria_db_aviso_cirurgia_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=aviso_cirurgia_sql_final)

{"timestamp": "2025-08-29T09:27:31", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection established to\n                srvawsdb002.cow7tj30bxpl.us-east-1.rds.amazonaws.com:3306/", "filename": "mariadb.py", "lineno": 27}
{"timestamp": "2025-08-29T09:27:31", "level": "INFO", "name": "app.services.database.mariadb", "message": "Executing MariaDB query...", "filename": "mariadb.py", "lineno": 67}
{"timestamp": "2025-08-29T09:27:31", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ Fetched 17 rows from MariaDB.", "filename": "mariadb.py", "lineno": 75}
{"timestamp": "2025-08-29T09:27:31", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection closed.", "filename": "mariadb.py", "lineno": 41}
{"timestamp": "2025-08-29T09:27:31", "level": "INFO", "name": "app.utils.db_operations", "message": "✅ Query executed successfully with 17 sample records", "filename": "db_operations.py", "lineno": 61}
{"

## getting a single table with the info we need

In [13]:
import pandas as pd

maria_db_procedimento_df_processed = maria_db_procedimento_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA", "procedure": "DS_PROCEDIMENTO"})
maria_db_procedimento_df_processed.drop(columns=["hospitalization_type"], inplace=True)
maria_db_procedimento_df_processed = maria_db_procedimento_df_processed[maria_db_procedimento_df_processed["DS_PROCEDIMENTO"].notna()]
maria_db_procedimento_df_processed.reset_index(drop=True, inplace=True)

autorizacao_no_need_cols = [col for col in oracle_db_autorizacao_df.columns if col not in ["CD_AVISO_CIRURGIA", "CD_GUIA", "CD_SENHA", "DS_GUIA_PATH"]]
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df.drop(columns=autorizacao_no_need_cols)
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df_processed[oracle_db_autorizacao_df_processed["DS_GUIA_PATH"].notna()]
oracle_db_autorizacao_df_processed.reset_index(drop=True, inplace=True)

finalizado_no_need_cols = [col for col in oracle_db_finalizado_df.columns if col not in ["CD_REGISTRO_VINCULADO", "DS_CONTATO"]]
oracle_db_finalizado_df_processed = oracle_db_finalizado_df.drop(columns=finalizado_no_need_cols)
oracle_db_finalizado_df_processed.rename(columns={"CD_REGISTRO_VINCULADO": "CD_AVISO_CIRURGIA"}, inplace=True)
oracle_db_finalizado_df_processed = oracle_db_finalizado_df_processed[oracle_db_finalizado_df_processed["DS_CONTATO"].notna()]
oracle_db_finalizado_df_processed.reset_index(drop=True, inplace=True)

maria_db_aviso_cirurgia_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA"}, inplace=True)
keep_cols = ["CD_AVISO_CIRURGIA", "health_insurance_name"]
maria_db_aviso_cirurgia_df = maria_db_aviso_cirurgia_df[keep_cols]
maria_db_aviso_cirurgia_df.reset_index(drop=True, inplace=True)

In [14]:
merged_df_autorizacao = pd.merge(
    maria_db_procedimento_df_processed,
    oracle_db_autorizacao_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

merged_df_autorizacao_final = pd.merge(
    merged_df_autorizacao,
    oracle_db_finalizado_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

final_extracted_df = pd.merge(
    merged_df_autorizacao_final,
    maria_db_aviso_cirurgia_df,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

In [15]:
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name
0,868647,"[{""code"":30101522,""description"":""EXTENSOS FERI...",19796704.0,J6EFDX0,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO


In [16]:
# entering the pdf link, downloading the pdf, adding as another col
import requests
import numpy as np
def fetch_pdf_bytes(url):
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200 and 'application/pdf' in response.headers.get('content-type', ''):
            return response.content
        else:
            return np.nan
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return np.nan

# Apply to all links in DS_GUIA_PATH
final_extracted_df['DS_PDF_BYTES'] = final_extracted_df['DS_GUIA_PATH'].apply(fetch_pdf_bytes)
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES
0,868647,"[{""code"":30101522,""description"":""EXTENSOS FERI...",19796704.0,J6EFDX0,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...


In [17]:
# Create a new column with clickable links for DS_GUIA_PATH
def make_clickable(url):
    if pd.notna(url):
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""
final_extracted_df['DS_GUIA_PATH_CLICKABLE'] = final_extracted_df['DS_GUIA_PATH'].apply(make_clickable)
from IPython.display import display, HTML
display(HTML(final_extracted_df[['DS_GUIA_PATH', 'DS_GUIA_PATH_CLICKABLE']].head(10).to_html(escape=False)))

,DS_GUIA_PATH,DS_GUIA_PATH_CLICKABLE
0,https://cdns.overmind.ai/autorizacao-bradesco-122539284-1756260183395.pdf,https://cdns.overmind.ai/autorizacao-bradesco-122539284-1756260183395.pdf


# part 2: inputing data to technique, observing the outputed data

In [18]:
# trying to use the mudular code to get the workflow feeling
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.utils.process_images import process_blobs
from app.utils.textract_service import TextractPDFAnalyzer
from app.utils.llm_service import AnthropicLLMService
from app.utils.aws_services_handler import create_boto3_client
from app.utils.config import load_config, AppConstants
from app.utils.logger import get_logger
from app.utils.system_prompts.autorizacao_prompt import Prompts


logger = get_logger(name=__name__)
config_vars = load_config()

In [19]:
app_constants = AppConstants()
textract_client = create_boto3_client("textract", config_vars)
bedrock_client = create_boto3_client("bedrock-runtime", config_vars)
s3_client = create_boto3_client("s3", config_vars)
bucket_name = "autorizacoes"

# Initialize the TextractPDFAnalyzer
pdf_analyzer = TextractPDFAnalyzer(
    textract_client=textract_client,
    s3_client=s3_client,
    bucket_name="autorizacoes-textract",
    bucket_folder="guias-pdf"
)

autorizacao_prompt = Prompts.autorizacao_extraction_prompt


llm_instance = AnthropicLLMService(
    model_id=app_constants.BEDROCK_DEFAULT_MODEL_ID,
    model_version=app_constants.BEDROCK_DEFAULT_MODEL_VERSION,
    client=bedrock_client,
    system_prompt=autorizacao_prompt,
    max_tokens=app_constants.MAX_TOKENS,
    temperature=app_constants.TEMPERATURE,
    budget_tokens=app_constants.BUDGET_TOKENS
)

{"timestamp": "2025-08-29T09:27:36", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-29T09:27:36", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-08-29T09:27:36", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-29T09:27:36", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-08-29T09:27:36", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente S3 para a região: us-east-1...

In [20]:
# llm processing

from typing import Dict, Tuple
# Process rows from the dataframe 
def llm_extraction(df: pd.DataFrame, limit=None) -> Tuple[Dict, Dict]:
    """
    Extract data from PDFs using Textract and process with LLM.
    
    Args:
        df (pd.DataFrame): DataFrame containing PDF information
        limit (int, optional): Limit number of rows to process
        
    Returns:
        Tuple[Dict, Dict]: LLM results and Textract results dictionaries
    """
    if limit:
        df = df[:limit]
    textract_results = {}
    llm_results = {}
    for index, row in df.iterrows():
        pdf_bytes = row['DS_PDF_BYTES']
        file_id = str(row['CD_AVISO_CIRURGIA'])
        
        if pd.isna(pdf_bytes):
            print(f"Skipping row {index}: No PDF bytes available")
            continue
        
        # Extract forms data using Textract
        forms_data = pdf_analyzer.extract_forms_data_from_pdf(
            pdf_bytes=pdf_bytes,
            file_id=file_id
        )
        textract_results[file_id] = forms_data
        
        
        # Process with LLM
        forms_data_str = str(forms_data)
        llm_response = llm_instance.invoke_model(input_str=forms_data_str)
        llm_results[file_id] = llm_response
    return llm_results, textract_results

def llm_dict_to_final_df(llm_results: Dict, textract_results: Dict, df: pd.DataFrame, limit=None) -> pd.DataFrame:
    """
    Convert LLM and Textract results dictionaries to final merged DataFrame.
    
    Args:
        llm_results (Dict): Dictionary with LLM processing results
        textract_results (Dict): Dictionary with Textract extraction results
        df (pd.DataFrame): Original DataFrame to merge with
        limit (int, optional): Limit number of rows from original DataFrame
        
    Returns:
        pd.DataFrame: Merged DataFrame with LLM and Textract results
    """
    if limit:
        df = df[:limit]

    llm_results_df = pd.DataFrame.from_dict(llm_results, orient="index")
    llm_results_df.index.name = "CD_AVISO_CIRURGIA"
    llm_results_df = llm_results_df.reset_index()

    # Expand LLM results into separate columns
    llm_expanded_df = pd.json_normalize(llm_results_df.iloc[:, 1:].to_dict('records'))
    llm_expanded_df['CD_AVISO_CIRURGIA'] = llm_results_df['CD_AVISO_CIRURGIA']

    # Add suffix to distinguish LLM columns
    llm_expanded_df = llm_expanded_df.add_suffix('_llm').rename(columns={'CD_AVISO_CIRURGIA_llm': 'CD_AVISO_CIRURGIA'})


    # add the full llm results as a new column, maping the llm dict to the CD_AVISO_CIRURGIA
    llm_expanded_df['llm_full_results'] = llm_expanded_df['CD_AVISO_CIRURGIA'].map(llm_results)
    # Convert CD_AVISO_CIRURGIA to int to match the original dataframe type
    llm_expanded_df['CD_AVISO_CIRURGIA'] = llm_expanded_df['CD_AVISO_CIRURGIA'].astype(int)
    # Map textract results based on CD_AVISO_CIRURGIA values
    # Convert the keys in textract_results to int for proper mapping
    textract_results_int_keys = {int(k): v for k, v in textract_results.items()}
    llm_expanded_df['textract_results'] = llm_expanded_df['CD_AVISO_CIRURGIA'].map(textract_results_int_keys)

    # Ensure original df has the correct data type
    df = df.copy()
    df['CD_AVISO_CIRURGIA'] = df['CD_AVISO_CIRURGIA'].astype(int)

    # Merge df with llm results
    merged_df = df.merge(
        llm_expanded_df,
        on='CD_AVISO_CIRURGIA',
        how='inner'
    )
    # Inner join with original dataframe
    return merged_df

In [21]:
llm_results, textract_results = llm_extraction(df=final_extracted_df)
llm_results_df_run = llm_dict_to_final_df(llm_results=llm_results, textract_results=textract_results, df=final_extracted_df)

print(f"Processing completed! Created textract_eval_df with {len(llm_results_df_run)} rows")
print(f"LLM columns added: {[col for col in llm_results_df_run.columns if col.endswith('_llm')]}")
llm_results_df_run.head()



{"timestamp": "2025-08-29T09:27:37", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF uploaded to S3: s3://autorizacoes-textract/guias-pdf/textract_pdfs/868647_20250829_092736_f4af60a8.pdf", "filename": "textract_service.py", "lineno": 177}
{"timestamp": "2025-08-29T09:27:38", "level": "INFO", "name": "app.utils.textract_service", "message": "Started PDF analysis with JobId: e4f6516da1600dba2e8586ef3411c472313c688dabf5938e8ae66e7fe36b59a7", "filename": "textract_service.py", "lineno": 210}
{"timestamp": "2025-08-29T09:27:39", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF analysis in progress for JobId: e4f6516da1600dba2e8586ef3411c472313c688dabf5938e8ae66e7fe36b59a7", "filename": "textract_service.py", "lineno": 264}
{"timestamp": "2025-08-29T09:27:44", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF analysis in progress for JobId: e4f6516da1600dba2e8586ef3411c472313c688dabf5938e8ae66e7fe36b59a7", "filename": "textra

Processing completed! Created textract_eval_df with 1 rows
LLM columns added: ['procedimentos_autorizados_llm', 'paciente_llm', 'codigo_autorizado_llm', 'senha_llm', 'validade_senha_llm', 'data_solicitacao_llm', 'observacoes_opme_llm', 'observacoes_gerais_llm', 'profissional_solicitante_llm']


,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES,DS_GUIA_PATH_CLICKABLE,procedimentos_autorizados_llm,paciente_llm,codigo_autorizado_llm,senha_llm,validade_senha_llm,data_solicitacao_llm,observacoes_opme_llm,observacoes_gerais_llm,profissional_solicitante_llm,llm_full_results,textract_results
0,868647,"[{""code"":30101522,""description"":""EXTENSOS FERI...",19796704.0,J6EFDX0,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[FERIMENTOS,CICAT OU TUMORES EXC E RET CUT DA ...",MATHEUS GOMES TONUSSI,30101522 x 2,J6EFDX0,None,13/08/2025,Pedido sem OPME,None,ANDRE VILLANI CORREA MAFRA,"{'procedimentos_autorizados': ['FERIMENTOS,CIC...","{'forms': {'16 Número do Conselho': '39920', '..."


In [22]:
llm_results_df_run.loc[0, 'llm_full_results']

{'procedimentos_autorizados': ['FERIMENTOS,CICAT OU TUMORES EXC E RET CUT DA REG'],
 'paciente': 'MATHEUS GOMES TONUSSI',
 'codigo_autorizado': '30101522 x 2',
 'senha': 'J6EFDX0',
 'validade_senha': None,
 'data_solicitacao': '13/08/2025',
 'observacoes_opme': 'Pedido sem OPME',
 'observacoes_gerais': None,
 'profissional_solicitante': 'ANDRE VILLANI CORREA MAFRA'}

In [23]:
llm_results_df = llm_results_df_run.copy()
llm_results_df.loc[0, 'CD_AVISO_CIRURGIA']

np.int64(868647)

In [24]:
# value of the col DS_PROCEDIMENTO of the 4 row:
llm_results_df.loc[0, 'DS_PROCEDIMENTO']

'[{"code":30101522,"description":"EXTENSOS FERIM-CICATRIZES OU TUMORES EXC","isMain":true,"pro_fat_id":"30101522","procedure_code":30101522,"quantity":2,"surgery_id":604}]'

In [26]:
# value of the col ds_contato of the 4 row:
llm_results_df.loc[0, 'DS_CONTATO']

'\n      Procedimento Autorizado\n      Paciente: MATHEUS GOMES TONUSSI\n      CÓDIGO AUTORIZADO:\n      01009012 x 1\n,30101522 x 2\n\n      \n      SENHA: J6EFDX0\n      VALIDADE DA SENHA: 09/02/2026\n      OBSERVAÇÕES OPME: Pedido sem OPME\n      \n      NOME USUÁRIO FINALIZOU O PEDIDO: ANA\n    '

In [28]:
llm_results_df.loc[0, 'codigo_autorizado_llm']

'30101522 x 2'

In [29]:
# Create a new column with clickable links for DS_GUIA_PATH
def make_clickable(url):
    if pd.notna(url):
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""
llm_results_df['DS_GUIA_PATH_CLICKABLE'] = llm_results_df['DS_GUIA_PATH'].apply(make_clickable)
from IPython.display import display, HTML
display(HTML(llm_results_df[['CD_AVISO_CIRURGIA','DS_GUIA_PATH', 'DS_GUIA_PATH_CLICKABLE']].head(10).to_html(escape=False)))

,CD_AVISO_CIRURGIA,DS_GUIA_PATH,DS_GUIA_PATH_CLICKABLE
0,868647,https://cdns.overmind.ai/autorizacao-bradesco-122539284-1756260183395.pdf,https://cdns.overmind.ai/autorizacao-bradesco-122539284-1756260183395.pdf


# Part 3: comparing the data from the llm with the field DS_PROCEDIMENTO, generating a new field validacao_status with the values "match" or "not match"

The new fields cols would be  validacao_status_codigo_procedimento, validacao_status_descricao_procedimento


In [30]:
# Part 3: comparing the data from the llm with the field DS_PROCEDIMENTO, generating a new field validacao_status with the values "match" or "not match"

# The new fields cols would be  validacao_status_codigo_procedimento, validacao_status_descricao_procedimento
# example of the field DS_PROCEDIMENTO:
# '[{"code":30205050,"description":"AMIGDALECTOMIA DAS PLATINAS","isMain":true,"pro_fat_id":"30205050","procedure_code":30205050,"quantity":1,"surgery_id":1227},{"code":30205271,"description":"ADENOIDECTOMIA POR VIDEOENDOSCOPIA","isMain":false,"pro_fat_id":"30205271","procedure_code":30205271,"quantity":1,"surgery_id":5974},{"code":30205069,"description":"AMIDALECTOMIA LINGUAL","isMain":false,"pro_fat_id":"30205069","procedure_code":30205069,"quantity":1,"surgery_id":1236},{"code":30501458,"description":"TURBINECTOMIA OU TURBINOPLASTIA - UNILATERAL","isMain":false,"pro_fat_id":"30501458","procedure_code":30501458,"quantity":2,"surgery_id":1191}]'
# example of the field 'codigo_autorizado_llm':
# '30205050 x 1, 30205069 x 1, 30205271 x 1, 30501458 x 2'




import pandas as pd
import json

import pandas as pd
import json
from typing import List, Tuple

def validate_codigo_procedimento(
    ds_procedimento_col: List[str], 
    codigo_autorizado_llm_col: List[str]
) -> Tuple[List[str], List[str]]:
    """
    Compares DS_PROCEDIMENTO and codigo_autorizado_llm columns to generate validation status and explanations.

    Args:
        ds_procedimento_col (List[str]): List of DS_PROCEDIMENTO JSON strings.
        codigo_autorizado_llm_col (List[str]): List of LLM output strings (e.g., '30205050 x 1, 30205069 x 1').

    Returns:
        Tuple[List[str], List[str]]: Tuple containing:
            - List of validation statuses ("match" or "not match") for each row
            - List of explanations describing what doesn't match
    """
    def parse_ds_procedimento(json_str):
        try:
            data_list = json.loads(json_str)
            return set(
                (str(item.get('procedure_code', item.get('code'))), str(item['quantity']))
                for item in data_list
            )
        except (json.JSONDecodeError, TypeError):
            return set()

    def parse_llm_data(llm_str):
        if pd.isna(llm_str):
            return set()
        try:
            items = llm_str.split(', ')
            return set(
                (item.split(' x ')[0], item.split(' x ')[1])
                for item in items
            )
        except IndexError:
            return set()

    def compare_data_with_explanation(ds_set, llm_set):
        if ds_set.issubset(llm_set):
            return "correspondente", "Todos os procedimentos de DS_PROCEDIMENTO estão presentes na saída do LLM"
        
        # Find missing procedures
        missing_in_llm = ds_set - llm_set
        extra_in_llm = llm_set - ds_set
        
        explanations = []
        
        if missing_in_llm:
            missing_codes = [f"código {code} x {qty}" for code, qty in missing_in_llm]
            explanations.append(f"Ausente no LLM: {', '.join(missing_codes)}")
        
        if extra_in_llm:
            extra_codes = [f"código {code} x {qty}" for code, qty in extra_in_llm]
            explanations.append(f"Extra no LLM: {', '.join(extra_codes)}")
        
        return "não correspondente", "; ".join(explanations)

    results = [
        compare_data_with_explanation(
            parse_ds_procedimento(ds_proc), 
            parse_llm_data(llm_val)
        )
        for ds_proc, llm_val in zip(ds_procedimento_col, codigo_autorizado_llm_col)
    ]
    
    statuses = [result[0] for result in results]
    explanations = [result[1] for result in results]
    
    return statuses, explanations



In [31]:
# Get both validation status and explanation
validation_status, validation_explanation = validate_codigo_procedimento(
    llm_results_df['DS_PROCEDIMENTO'].tolist(),
    llm_results_df['codigo_autorizado_llm'].tolist()
)

# Add both columns to the dataframe
llm_results_df['validacao_status_codigo_procedimento'] = validation_status
llm_results_df['validacao_explanation_codigo_procedimento'] = validation_explanation

llm_results_df.head()

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES,DS_GUIA_PATH_CLICKABLE,procedimentos_autorizados_llm,...,senha_llm,validade_senha_llm,data_solicitacao_llm,observacoes_opme_llm,observacoes_gerais_llm,profissional_solicitante_llm,llm_full_results,textract_results,validacao_status_codigo_procedimento,validacao_explanation_codigo_procedimento
0,868647,"[{""code"":30101522,""description"":""EXTENSOS FERI...",19796704.0,J6EFDX0,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[FERIMENTOS,CICAT OU TUMORES EXC E RET CUT DA ...",...,J6EFDX0,None,13/08/2025,Pedido sem OPME,None,ANDRE VILLANI CORREA MAFRA,"{'procedimentos_autorizados': ['FERIMENTOS,CIC...","{'forms': {'16 Número do Conselho': '39920', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...


In [32]:
## add the 

In [33]:
llm_results_df.loc[0, 'DS_PROCEDIMENTO']

'[{"code":30101522,"description":"EXTENSOS FERIM-CICATRIZES OU TUMORES EXC","isMain":true,"pro_fat_id":"30101522","procedure_code":30101522,"quantity":2,"surgery_id":604}]'

In [34]:
llm_results_df.loc[0, 'codigo_autorizado_llm']

'30101522 x 2'

In [35]:
llm_results_df.loc[0, 'validacao_explanation_codigo_procedimento']

'Todos os procedimentos de DS_PROCEDIMENTO estão presentes na saída do LLM'

In [36]:
llm_results_df.loc[0, 'validade_senha_llm']

In [37]:
llm_results_df.loc[0, 'textract_results']

{'forms': {'16 Número do Conselho': '39920',
  '14 Nome do Profissional Solicitante': 'ANDRE VILLANI CORREA MAFRA',
  '20 Nome do Hospital/ Local Solicitado': 'HOSPITAL MATER DEI',
  '19 Código na Operadora / CNPJ': '426512',
  '17 UF': 'MG',
  '15 Conselho Profissional': 'CRM',
  '1 Registro ANS': '005711',
  '41 Tipo da Acomodação Autorizada': 'QUARTO PARTICULAR',
  '25 Qtde. Diárias': '1',
  '9 Atendimento a RN': 'Não',
  '3 Número da Guia Atribuído pela Operadora': '122539284',
  '34 Tabela': '16',
  '23 -Tipo de': '2',
  '38 Qtde. Aut.': '2',
  '7 Número da Carteira': '861554400043019',
  '35 Código do Procedimento ou Item Assistencial': '30101522',
  '24 Regime de Internação': 'HOSPITAL-DIA',
  '22 Caráter do': 'ELETIVO',
  '5 Senha': 'J6EFDX0',
  '37 Qtde. Solic.': '2',
  '40 Qtde. Diárias Autorizadas': '0',
  '39 Data Provável da Admissão': '14/08/2025',
  '21 Data Sugerida para Internação (Real)': '14/08/2025',
  '10 Nome': 'MATHEUS GOMES TONUSSI',
  '13 Nome do Contratado': '

In [38]:
import pandas as pd
from datetime import timedelta
import numpy as np

# Replace string "None" with actual NaN values before processing
llm_results_df['validade_senha_llm'] = llm_results_df['validade_senha_llm'].replace('None', np.nan)

# Ensure date columns are in datetime format, coercing errors to NaT (Not a Time)
# This handles cases where the date might be None or in an invalid format
llm_results_df['data_solicitacao_llm'] = pd.to_datetime(llm_results_df['data_solicitacao_llm'], errors='coerce')
llm_results_df['validade_senha_llm'] = pd.to_datetime(llm_results_df['validade_senha_llm'], errors='coerce')

# Define the condition for the update
condition = (
    (llm_results_df['health_insurance_name'] == 'BRADESCO') &
    (llm_results_df['validade_senha_llm'].isna()) &
    (llm_results_df['data_solicitacao_llm'].notna())
)

# Apply the update where the condition is true
llm_results_df.loc[condition, 'validade_senha_llm'] = llm_results_df.loc[condition, 'data_solicitacao_llm'] + timedelta(days=180)

# transform the dates back to str, but not like this: "'2026-02-09'", but like 09-02-2026"
llm_results_df['data_solicitacao_llm'] = llm_results_df['data_solicitacao_llm'].dt.strftime('%d-%m-%Y')
llm_results_df['validade_senha_llm'] = llm_results_df['validade_senha_llm'].dt.strftime('%d-%m-%Y')

llm_results_df.head()


/tmp/ipykernel_2281/652204823.py:10: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  llm_results_df['data_solicitacao_llm'] = pd.to_datetime(llm_results_df['data_solicitacao_llm'], errors='coerce')


,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES,DS_GUIA_PATH_CLICKABLE,procedimentos_autorizados_llm,...,senha_llm,validade_senha_llm,data_solicitacao_llm,observacoes_opme_llm,observacoes_gerais_llm,profissional_solicitante_llm,llm_full_results,textract_results,validacao_status_codigo_procedimento,validacao_explanation_codigo_procedimento
0,868647,"[{""code"":30101522,""description"":""EXTENSOS FERI...",19796704.0,J6EFDX0,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[FERIMENTOS,CICAT OU TUMORES EXC E RET CUT DA ...",...,J6EFDX0,09-02-2026,13-08-2025,Pedido sem OPME,None,ANDRE VILLANI CORREA MAFRA,"{'procedimentos_autorizados': ['FERIMENTOS,CIC...","{'forms': {'16 Número do Conselho': '39920', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...


In [39]:
# get all the llm columns that ends with _llm, so we can add a value like this:
"""  {
    "procedimentos_autorizados": [
      "AMIGDALECTOMIA DAS PALATINAS",
      "AMIGDALECTOMIA LINGUAL",
      "UVULOPALATOFARINGOPLASTIA",
      "CORNETO INFERIOR - CAUTERIZACAO LINEAR UNILAT",
      "SEPTOPLASTIA SEM VIDEO",
      "TURBINECTOMIA OU TURBINOPLASTIA UNILATERAL",
      "ANTROSTOMIA MAXILAR INTRANASAL POR VIDEOENDOSCOPIA",
      "ETMOIDECTOMIA INTRANASAL POR VIDEOENDOSCOPIA",
      "SINUSOTOMIA ESFENOIDAL POR VIDEOENDOSCOPIA",
      "SINUSOTOMIA FRONTAL INTRANASAL POR VIDEOENDOSCOPIA"
    ],
    "paciente": "DIONE RAIMUNDO CARVALHO PINTO",
    "codigo_autorizado": "30205050 x 1, 30205069 x 1, 30205247 x 1, 30501067 x 2, 30501369 x 1, 30501458 x 2, 30502292 x 0, 30502314 x 2, 30502349 x 0, 30502357 x 0",
    "senha": "J5VEWF9",
    "validade_senha": null,
    "data_solicitacao": "11/07/2025",
    "observacoes_opme": "Pedido sem OPME",
    "observacoes_gerais": "Solicitacao de autorizacao",
    "profissional_solicitante": "LUCAS EDUARDO DE OLIVEIRA"
  }"""

llm_columns = [col for col in llm_results_df.columns if col.endswith('_llm')]
llm_columns


['procedimentos_autorizados_llm',
 'paciente_llm',
 'codigo_autorizado_llm',
 'senha_llm',
 'validade_senha_llm',
 'data_solicitacao_llm',
 'observacoes_opme_llm',
 'observacoes_gerais_llm',
 'profissional_solicitante_llm']

In [40]:
llm_results_df["llm_full_results"] = llm_results_df.apply(lambda row: {
    "procedimentos_autorizados": row.get("procedimentos_autorizados_llm", []),
    "paciente": row.get('paciente_llm', ""),
    "codigo_autorizado": row.get("codigo_autorizado_llm", ""),
    "senha": row.get("senha_llm", ""),
    "validade_senha": row.get("validade_senha_llm", ""),
    "data_solicitacao": row.get("data_solicitacao_llm", ""),
    "observacoes_opme": row.get("observacoes_opme_llm", ""),
    "observacoes_gerais": row.get("observacoes_gerais_llm", ""),
    "profissional_solicitante": row.get("profissional_solicitante_llm", "")
}, axis=1)

llm_results_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES,DS_GUIA_PATH_CLICKABLE,procedimentos_autorizados_llm,...,senha_llm,validade_senha_llm,data_solicitacao_llm,observacoes_opme_llm,observacoes_gerais_llm,profissional_solicitante_llm,llm_full_results,textract_results,validacao_status_codigo_procedimento,validacao_explanation_codigo_procedimento
0,868647,"[{""code"":30101522,""description"":""EXTENSOS FERI...",19796704.0,J6EFDX0,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[FERIMENTOS,CICAT OU TUMORES EXC E RET CUT DA ...",...,J6EFDX0,09-02-2026,13-08-2025,Pedido sem OPME,None,ANDRE VILLANI CORREA MAFRA,"{'procedimentos_autorizados': ['FERIMENTOS,CIC...","{'forms': {'16 Número do Conselho': '39920', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...


In [41]:
llm_results_df.loc[0, 'llm_full_results']

{'procedimentos_autorizados': ['FERIMENTOS,CICAT OU TUMORES EXC E RET CUT DA REG'],
 'paciente': 'MATHEUS GOMES TONUSSI',
 'codigo_autorizado': '30101522 x 2',
 'senha': 'J6EFDX0',
 'validade_senha': '09-02-2026',
 'data_solicitacao': '13-08-2025',
 'observacoes_opme': 'Pedido sem OPME',
 'observacoes_gerais': None,
 'profissional_solicitante': 'ANDRE VILLANI CORREA MAFRA'}

In [42]:
# final processing response: 
keep_cols = ['CD_AVISO_CIRURGIA', 'llm_full_results', 'validacao_status_codigo_procedimento', 'validacao_explanation_codigo_procedimento']
llm_results_df = llm_results_df[keep_cols]
llm_results_df.head(3)

,CD_AVISO_CIRURGIA,llm_full_results,validacao_status_codigo_procedimento,validacao_explanation_codigo_procedimento
0,868647,"{'procedimentos_autorizados': ['FERIMENTOS,CIC...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...


Part 4: get the llm results back to excel tab autorizacoes

In [75]:
def merge_llm_results_to_excel(
    xls: pd.ExcelFile,
    llm_results_df: pd.DataFrame,
    output_excel_path: str,
    target_tabs: list = None
) -> None:
    """
    Merge LLM results back to specific Excel tabs and save to a new file.
    
    Args:
        xls (pd.ExcelFile): Original Excel file object
        llm_results_df (pd.DataFrame): DataFrame containing LLM results with CD_AVISO_CIRURGIA as key
        output_excel_path (str): Path where the updated Excel file should be saved
        target_tabs (list, optional): List of tab names to merge LLM results with. 
                                    If None, merges with ["Autorizados", "Pendentes"]
    
    Returns:
        None: Saves the updated Excel file to the specified path
    """
    if target_tabs is None:
        return
    
    # Dictionary to store processed dataframes
    processed_tabs = {}
    
    # Process each target tab
    for tab_name in target_tabs:
        if tab_name in xls.sheet_names:
            # Read the tab
            df_tab = pd.read_excel(xls, sheet_name=tab_name)
            
            # Merge with LLM results
            df_tab_final = df_tab.merge(llm_results_df, on="CD_AVISO_CIRURGIA", how="left")
            
            # Store the processed dataframe
            processed_tabs[tab_name] = df_tab_final
            
            print(f"📊 Processed {tab_name}: {len(df_tab_final)} rows, {len(df_tab_final[df_tab_final['llm_full_results'].notna()])} with LLM results")
        else:
            print(f"⚠️  Warning: Tab '{tab_name}' not found in Excel file")
    
    # Write all tabs back to Excel
    with pd.ExcelWriter(output_excel_path, engine="openpyxl", mode="w") as writer:
        for sheet in xls.sheet_names:
            if sheet in processed_tabs:
                # Write the processed tab with LLM results
                processed_tabs[sheet].to_excel(writer, sheet_name=sheet, index=False)
            else:
                # Write the original tab unchanged
                df_other = pd.read_excel(xls, sheet_name=sheet)
                df_other.to_excel(writer, sheet_name=sheet, index=False)
    
    print(f"✅ Successfully saved updated Excel file to: {output_excel_path}")



In [76]:
# Use the function to merge results for both Autorizados and Pendentes tabs
merge_llm_results_to_excel(
    xls=xls,
    llm_results_df=llm_results_df,
    output_excel_path="docs/excel/llm_output_enriquecido.xlsx",
    target_tabs=["Autorizados", "Pendentes"]
)

📊 Processed Autorizados: 282 rows, 1 with LLM results
📊 Processed Pendentes: 84 rows, 0 with LLM results
✅ Successfully saved updated Excel file to: docs/excel/llm_output_enriquecido.xlsx
✅ Successfully saved updated Excel file to: docs/excel/llm_output_enriquecido.xlsx


In [ ]:
# Example usage of the merge_llm_results_to_excel function

# Example 1: Merge LLM results only to Autorizados tab
merge_llm_results_to_excel(
    xls=xls,
    llm_results_df=llm_results_df,
    output_excel_path="docs/excel/autorizados_only_with_llm.xlsx",
    target_tabs=["Autorizados"]
)

# Example 2: Merge LLM results only to Pendentes tab
merge_llm_results_to_excel(
    xls=xls,
    llm_results_df=llm_results_df,
    output_excel_path="docs/excel/pendentes_only_with_llm.xlsx",
    target_tabs=["Pendentes"]
)

# Example 3: Merge to a custom tab (if it exists)
# merge_llm_results_to_excel(
#     xls=xls,
#     llm_results_df=llm_results_df,
#     output_excel_path="docs/excel/custom_tab_with_llm.xlsx",
#     target_tabs=["YourCustomTabName"]
# )

print("\n🔄 Function usage examples completed!")
print("📁 Files created:")
print("   - docs/excel/llm_output_enriquecido.xlsx (both tabs)")
print("   - docs/excel/autorizados_only_with_llm.xlsx (Autorizados only)")
print("   - docs/excel/pendentes_only_with_llm.xlsx (Pendentes only)")

upload back to s3